# Audiobook Studio - VoxCPM2 端到端全链路测试 (Kaggle V100 GPU)

## 目标
- 真实数据流: 文本解析 → LLM 情绪标注 → 声学映射 → TTS 合成 → 音频输出
- 不挂 Mock，使用真实配置跑通全流程，产出最终音频文件
- 验证架构真正可用

## 环境准备
1. 开启 GPU 加速器 (Settings -> GPU -> V100)
2. 依次运行所有单元格
3. 最终音频将保存到 `/kaggle/working/output/`

In [ ]:
!pip install -q torch torchaudio transformers accelerate huggingface_hub safetensors soundfile -q 2>&1 | tail -5

In [ ]:
import torch
import torch.nn as nn
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN: {torch.backends.cudnn.version()}")
else:
    print("CUDA not available - 请在 Kaggle 设置中开启 GPU (V100)")

In [ ]:
# 下载 VoxCPM2 模型 (使用镜像加速)
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"

from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="openbmb/VoxCPM2",
    local_dir="/kaggle/working/VoxCPM2",
    local_dir_use_symlinks=False
)
print(f"模型下载完成: {model_path}")

In [ ]:
# 检查模型结构
import os
model_path = "/kaggle/working/VoxCPM2"
for root, dirs, files in os.walk(model_path):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# VoxCPM2 推理测试
import torch
import torch.nn as nn

# 尝试加载模型
from transformers import AutoModel, AutoTokenizer

model_path = "/kaggle/working/VoxCPM2"

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/working/VoxCPM2",
    trust_remote_code=True
)

# 加载模型 (fp16 以节省显存)
model = AutoModel.from_pretrained(
    "/kaggle/working/VoxCPM2",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

model.eval()
print(f"模型加载完成，设备: {next(model.parameters()).device}")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
# 完整的推理流程封装
import torch
import torchaudio
import tempfile
import os
import soundfile as sf

class VoxCPM2Inference:
    def __init__(self, model_path="/kaggle/working/VoxCPM2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_path = model_path
        self.model = None
        self.tokenizer = None
        self._load_model()
    
    def _load_model(self):
        from transformers import AutoModel, AutoTokenizer
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_path,
            trust_remote_code=True
        )
        
        from transformers import AutoModel
        self.model = AutoModel.from_pretrained(
            self.model_path,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )
        self.model.eval()
        
    def synthesize(self, text: str, speaker_id: int = 0, temperature: float = 0.7) -> torch.Tensor:
        """合成语音，返回音频张量 (24kHz)"""
        # 编码文本
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        
        # 生成语音码
        with torch.no_grad():
            audio_tokens = self.model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=True,
                temperature=temperature,
                top_p=0.9,
                speaker_id=speaker_id,
            )
            
            # 解码为波形
            waveform = self.model.decode_audio(audio_tokens)
            
        return waveform
    
    def save_audio(self, audio_tensor, sample_rate: int, output_path: str):
        """保存音频文件"""
        torchaudio.save(output_path, audio_tensor.cpu(), sample_rate)
        return output_path

# 初始化
infer = VoxCPM2Inference()

# 测试文本 (来自 /Users/guwj/Desktop/AI_Lab/audiobook/input/test_story.txt)
test_texts = [
    "第一章 暗夜风暴前的裂痕。雷暴雨在倾盆而下，狂风肆虐地撕扯着古宅破旧的木窗。",
    "陆沉站在大厅中央，雨水顺着他漆黑的发丝不断淌下。陆沉：宋老，我再给您最后一次机会......把三年前那份档案交出来。",
    "宋老坐在轮椅上，缓缓转过身来。宋老：呵呵呵......陆沉啊陆沉，你真以为凭你一个人，就能从老夫手里拿走它？",
    "突然，大门被轰然撞开！顾清雪提着一把沾满泥水和鲜血的短刀冲了进来。顾清雪：别听他的！陆沉！他早就把档案烧了！他是骗你的！他在利用你！！",
    "第二章 绝崖边的生死摊牌。嘟——嘟——嘟——！刺耳的警报声瞬间响彻整栋大宅。",
    "顾清雪脚下一滑，整个人直接向悬崖边的裂口滑去！顾清雪：啊——！陆沉！救我！！",
    "陆沉没有半点犹豫，整个人如猎豹般扑了出去，在千钧一发之际，死死抓住了顾清雪冰冷的手腕！",
    "第三章 归途与未竟之谜。雨，渐渐小了。浓重的雾气笼罩着整片江面。",
    "陆沉脱下湿透的外套，轻轻披在坐在救护车后备箱上的顾清雪身上。陆沉：一切都结束了。宋老坠入了深江。",
    "顾清雪缓缓抬头，看着陆沉那张布满伤痕与疲惫却依然坚毅的脸，嘴角泛起一抹苦涩的笑容。",
    "陆沉从口袋里摸出一个半烧焦的金属储存卡，在阳光下散发着冰冷的光泽。陆沉：你说得对。因为......真相，才刚刚被我们握在手里。"
]

# 测试生成
os.makedirs("/kaggle/working/output", exist_ok=True)

for i, text in enumerate(test_texts):
    print(f"\nTesting {i+1}/{len(test_texts)}: {text[:50]}...")
    try:
        audio = infer.synthesize(text, speaker_id=i % 4)
        output = f"/kaggle/working/output/chapter_{i+1}.wav"
        infer.save_audio(audio, 24000, output)
        print(f"✅ 生成成功: {output}")
    except Exception as e:
        print(f"❌ 错误: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# 验证生成的音频文件
import os
for f in os.listdir("/kaggle/working/output"):
    if f.endswith(".wav"):
        path = f"/kaggle/working/output/{f}"
        import soundfile as sf
        info = sf.info(path)
        print(f"{f}: {info.duration:.2f}s, {info.samplerate}Hz, {info.channels}ch, {info.format}")

In [ ]:
# 显存占用总结
if torch.cuda.is_available():
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"GPU Max Memory Allocated: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")